# 03 - SHAP interpretability

Verifies that the attributions shown in the dashboard are real and that they
reconcile with the probability displayed next to them.

The key property: `shap_values.sum() + expected_value == raw_margin` for the exact
LightGBM booster that serves the prediction. The earlier version of this project
explained a *different* model from the one it served, so the bars did not add up to
anything.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "ml" / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "ml" / "src"))
DATA = ROOT / "ml" / "data" / "z_alizadeh_sani.xlsx"
MODELS = ROOT / "backend" / "app" / "models"
RESULTS = ROOT / "ml" / "results"
print("repo root:", ROOT)

In [ ]:
import joblib

models = joblib.load(MODELS / "cardiac_models.joblib")
metadata = json.loads((MODELS / "metadata.json").read_text())
thresholds = json.loads((MODELS / "thresholds.json").read_text())
presets = json.loads((MODELS / "demo_presets.json").read_text())

feature_order = metadata["raw_features"]
processed_names = metadata["processed_feature_names"]
print(f"{len(models)} models, {len(feature_order)} raw features -> {len(processed_names)} processed")
print("attribution engine:", metadata["shap_implementation"])


def tree_shap(pipe, frame):
    """TreeSHAP for one or many rows: (contributions, base_value).

    LightGBM's pred_contrib returns one column per transformed feature plus a
    final base-value column, and each row sums to that row's raw margin.
    """
    transformed = pipe[:-1].transform(frame)
    raw = np.asarray(pipe.named_steps["clf"].predict(transformed, pred_contrib=True))
    return raw[:, :-1], float(raw[0, -1])

## Additivity check

This is the test the old pipeline would have failed.

In [ ]:
from train import extract_targets, load_dataset

X, y = extract_targets(load_dataset(DATA))

for target, pipe in models.items():
    contrib, base = tree_shap(pipe, X)
    margin = pipe.named_steps["clf"].predict(pipe[:-1].transform(X), raw_score=True)
    residual = np.abs(contrib.sum(axis=1) + base - margin).max()
    print(f"{target:4s} base={base:+.4f}  max |shap_sum + base - margin| = {residual:.2e}")

Residuals at machine precision confirm the explainer and the served booster are the
same object.

## Global importance per vessel

Each vessel is driven by a different physiology, which is what makes the per-vessel
3D colouring meaningful rather than three copies of one number.

In [ ]:
import matplotlib.pyplot as plt


def source_feature(processed):
    """Map a transformed column name back to its original clinical feature."""
    name = processed.split("__", 1)[-1]
    if name in feature_order:
        return name
    cands = [f for f in feature_order if name.startswith(f)]
    return max(cands, key=len) if cands else name


fig, axes = plt.subplots(1, 4, figsize=(18, 5.5))
for ax, target in zip(axes, ["CAD", "LAD", "LCX", "RCA"]):
    contrib, _ = tree_shap(models[target], X)
    importance = {}
    for i, pname in enumerate(processed_names[:contrib.shape[1]]):
        src = source_feature(pname)
        importance[src] = importance.get(src, 0.0) + np.abs(contrib[:, i]).mean()
    top = sorted(importance.items(), key=lambda kv: kv[1], reverse=True)[:12][::-1]
    ax.barh([k for k, _ in top], [v for _, v in top], color="#2563eb")
    ax.set_title(f"{target}  mean |SHAP|", fontsize=11)
    ax.spines[["top", "right"]].set_visible(False)
fig.suptitle("Global attribution per target (log-odds margin space)")
fig.tight_layout()
plt.show()

## Patient-level explanation

The same computation the `/predict` endpoint performs, for the three-vessel demo
preset. These are the numbers the dashboard waterfall renders.

In [ ]:
case = max(presets, key=lambda p: p["vessels_involved"])
print(case["patient_id"], "|", case["label"], "| ground truth:", case["ground_truth"])

row = pd.DataFrame([{c: case["features"].get(c) for c in feature_order}], columns=feature_order)

for target in ["CAD", "LAD", "LCX", "RCA"]:
    pipe = models[target]
    prob = float(pipe.predict_proba(row)[0, 1])
    contrib, base = tree_shap(pipe, row)
    contrib = contrib.reshape(-1)
    order = np.argsort(np.abs(contrib))[::-1][:6]
    tuned = thresholds[target]["tuned"]
    flag = "STENOTIC" if prob >= tuned else "normal"
    print()
    print(f"{target}  p={prob:.3f}  (threshold {tuned:.3f}) -> {flag}")
    for i in order:
        src = source_feature(processed_names[i])
        arrow = "+" if contrib[i] > 0 else "-"
        value = str(row.iloc[0][src])
        print(f"   {arrow} {src:24s} = {value:>8s}   shap={contrib[i]:+.4f}")

## Why the dashboard shows two numbers

SHAP is additive in **log-odds margin** space, while clinicians read probabilities.
The API returns the calibrated probability as `risk_score` and the margin-space
attributions alongside it, with `shap_base_value` as the cohort baseline. The map
between them is monotonic, so the ranking the bars explain is the ranking displayed -
but the bars sum to the margin, not to the percentage, and the UI labels them as
such instead of implying they add up to the risk score.